# 🏎️ F1 Lap Time Predictor
## Notebook 1 — Exploratory Data Analysis

1. Load & inspect
2. Lap time distributions
3. Tyre degradation analysis
4. Weather effects
5. Driver & team pace analysis
6. Circuit characteristics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
ACCENT='#58a6ff'; DANGER='#f85149'; SUCCESS='#3fb950'; YELLOW='#f0a500'
COMPOUND_COLORS={'SOFT':'#f85149','MEDIUM':'#f0a500','HARD':'#e6edf3','INTER':'#3fb950','WET':'#58a6ff'}

df = pd.read_csv('../data/raw/f1_laps.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Dataset Overview ===')
print(f'Laps:     {len(df):,}')
print(f'Seasons:  {sorted(df["season"].unique())}')
print(f'Circuits: {df["circuit"].nunique()}')
print(f'Drivers:  {df["driver"].nunique()}')
print(f'Teams:    {df["team"].nunique()}')
print()
clean = df[(df['is_outlier']==0) & (df['is_pit_lap']==0)]
print(f'Clean laps: {len(clean):,}')
print(f'Avg lap time: {clean["lap_time_s"].mean():.2f}s')
clean.describe()

In [ ]:
# Tyre degradation by compound
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for ax in axes: ax.set_facecolor('#161b22')

for compound in ['SOFT','MEDIUM','HARD']:
    sub = clean[clean['compound']==compound]
    agg = sub.groupby('tyre_life')['lap_time_s'].mean()
    agg = agg[agg.index <= 40]
    norm = agg - agg.min()
    axes[0].plot(norm.index, norm.values, color=COMPOUND_COLORS[compound],
                 linewidth=2.5, label=compound, marker='o', markersize=3)

axes[0].set_xlabel('Tyre Age (laps)')
axes[0].set_ylabel('Delta from best (seconds)')
axes[0].set_title('Tyre Degradation by Compound', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.15)

# Deg rate by circuit
degs = []
for c in clean['circuit'].unique():
    for compound in ['SOFT','MEDIUM','HARD']:
        sub = clean[(clean['circuit']==c)&(clean['compound']==compound)]
        if len(sub) < 20: continue
        agg = sub.groupby('tyre_life')['lap_time_s'].mean()
        agg = agg[agg.index <= 25]
        if len(agg) > 5:
            slope = np.polyfit(agg.index, agg.values, 1)[0]
            degs.append({'circuit':c,'compound':compound,'deg_per_lap':slope})

deg_df = pd.DataFrame(degs)
med_deg = deg_df[deg_df['compound']=='MEDIUM'].set_index('circuit')['deg_per_lap'].sort_values(ascending=False)
axes[1].barh(med_deg.index, med_deg.values, color=[DANGER if v>0.06 else YELLOW if v>0.03 else SUCCESS for v in med_deg.values])
axes[1].set_xlabel('Degradation rate (s/lap) — MEDIUM compound')
axes[1].set_title('Circuit Degradation Rate (MEDIUM)', fontsize=12)
axes[1].grid(axis='x', alpha=0.15)

plt.tight_layout()
plt.savefig('../outputs/eda_tyre_degradation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# Team pace comparison
team_lap = clean.groupby(['circuit','team'])['lap_time_s'].median().reset_index()
best_per_circuit = team_lap.groupby('circuit')['lap_time_s'].min().reset_index()
best_per_circuit.columns = ['circuit','best_time']
team_lap = team_lap.merge(best_per_circuit, on='circuit')
team_lap['gap_to_best'] = team_lap['lap_time_s'] - team_lap['best_time']

team_avg = team_lap.groupby('team')['gap_to_best'].mean().sort_values()

fig, ax = plt.subplots(figsize=(10,5))
ax.set_facecolor('#161b22')
colors = [SUCCESS if i==0 else (YELLOW if i<3 else DANGER if i>7 else ACCENT) for i in range(len(team_avg))]
ax.barh(team_avg.index, team_avg.values, color=colors, alpha=0.9)
ax.set_xlabel('Average gap to fastest car (seconds)')
ax.set_title('Team Pace Ranking — Avg Gap to Leader', fontsize=12)
ax.axvline(0, color='#8b949e', linewidth=0.8)
ax.grid(axis='x', alpha=0.15)
for i, v in enumerate(team_avg.values):
    ax.text(v+0.01, i, f'+{v:.2f}s', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/eda_team_pace.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# Weather effects
fig, axes = plt.subplots(2, 2, figsize=(12,8))
for ax in axes.flatten(): ax.set_facecolor('#161b22')

# Track temp vs lap time
axes[0,0].scatter(clean['track_temp'], clean['lap_time_s'], alpha=0.05, s=2, color=ACCENT)
axes[0,0].set_xlabel('Track Temperature (°C)')
axes[0,0].set_ylabel('Lap Time (s)')
axes[0,0].set_title('Track Temp vs Lap Time', fontsize=11)

# Rain vs dry
for i, (rain, label, color) in enumerate([(0,'Dry',SUCCESS),(1,'Wet',ACCENT)]):
    subset = clean[clean['is_raining']==rain]['lap_time_s']
    axes[0,1].hist(subset, bins=40, alpha=0.65, color=color, label=label, density=True)
axes[0,1].set_title('Lap Time: Dry vs Wet', fontsize=11)
axes[0,1].legend()

# Humidity
axes[1,0].scatter(clean['humidity'], clean['lap_time_s'], alpha=0.05, s=2, color=YELLOW)
axes[1,0].set_xlabel('Humidity (%)')
axes[1,0].set_ylabel('Lap Time (s)')
axes[1,0].set_title('Humidity vs Lap Time', fontsize=11)

# Air temp distribution
for compound in ['SOFT','MEDIUM','HARD']:
    subset = clean[clean['compound']==compound]['air_temp']
    axes[1,1].hist(subset, bins=30, alpha=0.55, color=COMPOUND_COLORS[compound], label=compound, density=True)
axes[1,1].set_title('Air Temp Distribution by Compound Used', fontsize=11)
axes[1,1].legend(fontsize=9)

for ax in axes.flatten():
    ax.grid(alpha=0.12)

plt.suptitle('Weather Effects on F1 Lap Times', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/eda_weather.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# Correlation heatmap
cols = ['tyre_life','fuel_remaining_laps','track_temp','air_temp','humidity',
        'is_raining','safety_car_laps_ago','circuit_deg_factor','lap_number','lap_time_s']
corr = clean[cols].corr()

fig, ax = plt.subplots(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax, linewidths=0.3, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('../outputs/eda_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# Key EDA takeaways
print('=== Key EDA Findings ===')
for compound in ['SOFT','MEDIUM','HARD']:
    sub = clean[clean['compound']==compound]
    agg = sub.groupby('tyre_life')['lap_time_s'].mean()
    early = agg[agg.index <= 5].mean()
    late = agg[(agg.index >= 20)&(agg.index <= 25)].mean()
    if not np.isnan(late):
        print(f'{compound}: +{late-early:.2f}s degradation from lap 1-5 to lap 20-25')

print()
dry_mean = clean[clean['is_raining']==0]['lap_time_s'].mean()
wet_mean = clean[clean['is_raining']==1]['lap_time_s'].mean()
print(f'Wet vs Dry avg lap time difference: +{wet_mean-dry_mean:.2f}s')